In [2]:
!pip install faker

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   --------------- ------------------------ 0.8/2.0 MB 5.6 MB/s eta 0:00:01
   ------------------------------- -------- 1.6/2.0 MB 4.4 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 4.2 MB/s  0:00:00



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import pandas as pd
import numpy as np
from faker import Faker
fake = Faker()
np.random.seed(42)  # For reproducibility
# Example snippet - expand for full fraud simulation
accounts = [f'ACC_{i}' for i in range(5000)]
transactions = []
for _ in range(50000):
    transaction_id = fake.uuid4()
    fr = np.random.choice(accounts)
    to = np.random.choice(accounts)
    amt = np.random.lognormal(5, 1)  # Realistic amounts
    transactions.append({'transaction_id': transaction_id, 'from': fr, 'to': to, 'amount': amt, 'ts': fake.date_time()})
df = pd.DataFrame(transactions)
df.to_csv('transactions.csv', index=False)


In [21]:
import pandas as pd
import numpy as np
from collections import defaultdict
from datetime import datetime

# Load your data (adjust path as needed)
df = pd.read_csv('transactions.csv')  # Your 5 sample rows + more
df['ts'] = pd.to_datetime(df['ts'], format='mixed', dayfirst=True)

print("Dataset loaded:", df.shape)
print(df.head())


Dataset loaded: (50000, 5)
                         transaction_id      from        to       amount  \
0  2d861940-5ab2-4da2-9169-c91a3dfc347a   ACC_860  ACC_3772    44.799347   
1  f2dbf3e6-9876-422b-8ff1-4e1000a6a320  ACC_4426  ACC_3444  1263.522118   
2  eeeda4fe-433e-4456-9100-0f561216f871  ACC_3171  ACC_2919    83.023348   
3  c61889ce-d19c-470b-9f40-0d68850ec74f  ACC_1184  ACC_4555    87.779694   
4  511340fc-f111-49d3-ac3c-5052798c3bc5  ACC_3385  ACC_4117    12.947390   

                   ts  
0 1974-03-08 21:26:01  
1 1992-04-02 16:50:20  
2 2008-04-03 16:53:37  
3 1997-05-06 22:21:42  
4 1993-09-28 07:04:31  


In [ ]:
# 1. Extract unique accounts
accounts = sorted(set(df['from'].unique()) | set(df['to'].unique())) #combine both sender and receiver accounts
account_to_idx = {acc: i for i, acc in enumerate(accounts)} # Map accounts to indices for graph representation
n_accounts = len(accounts)
print(f"Unique accounts: {n_accounts}")

# 2. Build directed graph: sender -> [(receiver, amount), ...]
graph = defaultdict(list) #creates an empty list if a key doesn’t exist.
for _, row in df.iterrows(): 
    sender = row['from']
    receiver = row['to']
    amount = row['amount']
    graph[sender].append((receiver, amount))

print("Sample graph:")
for sender in list(graph.keys())[:3]:
    print(f"{sender}: {graph[sender][:2]}...")  # First 2 transactions


Unique accounts: 5000
Sample graph:
ACC_860: [('ACC_3772', 44.79934744907723), ('ACC_2003', 158.98719393775835)]...
ACC_4426: [('ACC_3444', 1263.522118251791), ('ACC_4656', 1582.4495977583238)]...
ACC_3171: [('ACC_2919', 83.02334771214421), ('ACC_2668', 290.8728459898823)]...


In [23]:
# 3. Degree and aggregate statistics
in_degree = defaultdict(int)
out_degree = defaultdict(int)
total_sent = defaultdict(float)
total_received = defaultdict(float)
transaction_count = defaultdict(int)

for _, row in df.iterrows():
    sender, receiver = row['from'], row['to']
    amt = row['amount']
    
    out_degree[sender] += 1
    in_degree[receiver] += 1
    total_sent[sender] += amt
    total_received[receiver] += amt
    transaction_count[sender] += 1
    transaction_count[receiver] += 1

# Convert to DataFrame for ML later
features = pd.DataFrame({
    'account': accounts,
    'in_degree': [in_degree.get(acc, 0) for acc in accounts],
    'out_degree': [out_degree.get(acc, 0) for acc in accounts],
    'total_sent': [total_sent.get(acc, 0) for acc in accounts],
    'total_received': [total_received.get(acc, 0) for acc in accounts],
    'trans_count': [transaction_count.get(acc, 0) for acc in accounts]
})

print("\nGraph Features (top 10 active accounts):")
print(features.nlargest(10, 'trans_count')[['account', 'in_degree', 'out_degree', 'total_sent']])



Graph Features (top 10 active accounts):
       account  in_degree  out_degree   total_sent
807   ACC_1724         22          19  4285.924328
2261  ACC_3032         19          20  3345.071380
1287  ACC_2156         19          17  3781.361025
2973  ACC_3674         19          17  3211.714399
4267  ACC_4839         14          22  5561.105231
1589  ACC_2428         25          10  1382.539841
3679  ACC_4309         20          15  3021.353268
4695   ACC_724         15          20  5316.539087
156   ACC_1138         14          20  3083.973447
692   ACC_1620         20          14  2700.595631


In [ ]:
# Step 2: Union-Find for Connected Components (Hard Links Simulation)
class UnionFind:
    def __init__(self, size):
        self.parent = list(range(size))
        self.rank = [0] * size
        self.size = [1] * size  # Track component size

    def find(self, p):
        if self.parent[p] != p:
            self.parent[p] = self.find(self.parent[p])
        return self.parent[p]

    def union(self, p, q):
        pp, pq = self.find(p), self.find(q)
        if pp == pq: return False
        # Union by rank + update size
        if self.rank[pp] < self.rank[pq]:
            self.parent[pp] = pq
            self.size[pq] += self.size[pp]
        elif self.rank[pp] > self.rank[pq]:
            self.parent[pq] = pp
            self.size[pp] += self.size[pq]
        else:
            self.parent[pq] = pp
            self.rank[pp] += 1
            self.size[pp] += self.size[pq]
        return True

# Map accounts to indices
account_to_idx = {acc: i for i, acc in enumerate(accounts)}
uf = UnionFind(len(accounts))

# Simulate hard links: Union accounts with similar activity/behavior (10% pairs)
# In real: use shared phone/email/IP
import random
high_activity = features[features['trans_count'] > features['trans_count'].quantile(0.8)]
for i in range(min(1000, len(high_activity)//2)):  # 1000 unions
    idx1, idx2 = random.sample(high_activity.index.tolist(), 2)
    uf.union(idx1, idx2)

# Get ALL Components + Sizes
components = defaultdict(list)
component_sizes = defaultdict(int)
for i, acc in enumerate(accounts):
    root = uf.find(i)
    components[root].append(acc)
    component_sizes[root] = uf.size[root]

print("\n" + "="*60)
print("CONNECTED COMPONENTS ANALYSIS")
print("="*60)
print(f"Total components: {len(components)}")
print(f"Largest component size: {max(component_sizes.values())}")
print(f"Average size: {np.mean(list(component_sizes.values())):.1f}")

# Top 5 Largest Clusters
top_clusters = sorted(component_sizes.items(), key=lambda x: x[1], reverse=True)[:5]
print("\nTOP 5 LARGEST CLUSTERS:")
for rank, (root, size) in enumerate(top_clusters, 1):
    sample_accounts = components[root][:5]  # First 5 accounts
    print(f"{rank}. Component {root}: {size} accounts")
    print(f"   Sample: {sample_accounts}")



🧠 CONNECTED COMPONENTS ANALYSIS
Total components: 4613
Largest component size: 116
Average size: 1.1

🏆 TOP 5 LARGEST CLUSTERS:
1. Component 4860: 116 accounts
   Sample: ['ACC_1058', 'ACC_1068', 'ACC_1096', 'ACC_1098', 'ACC_1160']
2. Component 3858: 25 accounts
   Sample: ['ACC_1091', 'ACC_1252', 'ACC_155', 'ACC_1724', 'ACC_1852']
3. Component 1904: 24 accounts
   Sample: ['ACC_1279', 'ACC_1318', 'ACC_1558', 'ACC_1747', 'ACC_1753']
4. Component 2778: 13 accounts
   Sample: ['ACC_135', 'ACC_1573', 'ACC_2431', 'ACC_2492', 'ACC_2779']
5. Component 199: 11 accounts
   Sample: ['ACC_1138', 'ACC_1177', 'ACC_1449', 'ACC_2073', 'ACC_2500']


In [ ]:
# Flag Suspicious Clusters (for fraud)
suspicious_threshold = features['trans_count'].quantile(0.95)
suspicious_clusters = [root for root, size in component_sizes.items() 
                      if component_sizes[root] > 50 or np.mean([features[features['account']==acc]['trans_count'].iloc[0] for acc in components[root]]) > suspicious_threshold]

print(f"\nSuspicious clusters: {len(suspicious_clusters)}")
for root in suspicious_clusters[:4]:
    print(f"  - Cluster {root}: {len(components[root])} accts, high activity")



🚨 Suspicious clusters: 77
  - Cluster 3194: 3 accts, high activity
  - Cluster 4860: 116 accts, high activity
  - Cluster 102: 1 accts, high activity
  - Cluster 3858: 25 accts, high activity
